### NLP Project
### Author: Chris Marcus    
### Date: 8/14/24



## **Problem Statement**

### Business Context

In today's dynamic business landscape, organizations are increasingly recognizing the pivotal role customer feedback plays in shaping the trajectory of their products and services. The ability to swiftly and effectively respond to customer input not only fosters enhanced customer experiences but also serves as a catalyst for growth, prolonged customer engagement, and the nurturing of lifetime value relationships. As a dedicated Product Manager or Product Analyst, staying attuned to the voice of your customers is not just a best practice; it's a strategic imperative.

While your organization may be inundated with a wealth of customer-generated feedback and support tickets, your role entails much more than just processing these inputs. To make your efforts in managing customer experience and expectations truly impactful, you need a structured approach – a method that allows you to discern the most pressing issues, set priorities, and allocate resources judiciously. One of the most effective strategies at your disposal is to harness the power of Support Ticket Categorization.


### Objective

Develop an advanced support ticket categorization system that accurately classifies incoming tickets, assigns relevant tags based on their content, implements mechanisms and generate the first response based on the sentiment for prioritizing tickets for prompt resolution.


## **Installing and Importing Necessary Libraries and Dependencies**

In [247]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used

!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 240.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 229.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 219.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.3.1+cu121 requires nvidia-cublas-cu12==12.1.3.1; platform_system == "Linux" and platform_machine == "x86_64", which is not installed.
torch 2.3.1+cu121 requires nvidia-cuda-cupti-cu12==12.1.105; platform_system == "Linux" and platform_machine == "x86_64", which is not installed.
torch 2.3.1+cu121 requires nvidia-cuda-nvrtc-cu12==12.1.105; platform_system == "

In [248]:
# For downloading the models from HF Hub
!pip install huggingface_hub==0.20.3 pandas==1.5.3 -q

In [249]:
# Function to download the model from the Hugging Face model hub
from huggingface_hub import hf_hub_download

# Importing the Llama class from the llama_cpp module
from llama_cpp import Llama

# Importing the json module
import json

# for loading and manipulating data
import pandas as pd

# for time computations
import time

## **Loading the Data**

In [250]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [251]:
# Complete the code to read the CSV file.
data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Support_ticket_text_data_mid_term.csv")

## **Data Overview**

### Checking the first 5 rows of the data

In [252]:
# Complete the code to check the first 5 rows of the data
data.head()

,support_tick_id,support_ticket_text
0,ST2023-006,My internet connection has significantly slowe...
1,ST2023-007,Urgent help required! My laptop refuses to sta...
2,ST2023-008,I've accidentally deleted essential work docum...
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...
4,ST2023-010,"My smartphone battery is draining rapidly, eve..."


### Checking the shape of the data

In [253]:
# Complete the code to check the shape of the data
data.shape

(21, 2)

### Checking the missing values in the data

In [254]:
# Complete the code to check for missing values in the data
data.isnull().sum()

,0
support_tick_id,0
support_ticket_text,0


## **Model Building**

### Loading the model (Mistral)

In [255]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [256]:
# Using hf_hub_download to download a model from the Hugging Face model hub
# The repo_id parameter specifies the model name or path in the Hugging Face repository
# The filename parameter specifies the name of the file to download
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

In [257]:
llm = Llama(
    model_path=model_path,
    n_ctx=1024,
)

AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


### Utility functions

In [258]:
# defining a function to parse the JSON output from the model
def OLD_extract_json_data(json_str):
    try:
        # Find the indices of the opening and closing curly braces
        json_start = json_str.find('{')
        json_end = json_str.rfind('}')

        if json_start != -1 and json_end != -1:
            extracted_sentiment = json_str[json_start:json_end + 1]  # Extract the JSON object
            data_dict = json.loads(extracted_sentiment)
            return data_dict
        else:
            print(f"Warning: JSON object not found in response: {json_str}")
            return {}
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return {}

In [259]:
import json

def extract_json_data(json_str):
    try:
        # Find the indices of the opening and closing curly braces
        json_start = json_str.find('{')
        json_end = json_str.rfind('}')

        if json_start != -1 and json_end != -1:
            extracted_sentiment = json_str[json_start:json_end + 1]  # Extract the JSON object
            print(f"Extracted JSON: {extracted_sentiment}")  # Debugging line
            data_dict = json.loads(extracted_sentiment)
            return data_dict
        else:
            print(f"Warning: JSON object not found in response: {json_str}")
            return {}
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        print(f"Problematic JSON: {extracted_sentiment}")  # Show what was being parsed
        return {}


In [260]:
def extract_category(model_response):
    if 'hardware' in model_response.lower():
        return 'Hardware'
    elif 'software' in model_response.lower():
        return 'Software'
    elif 'network' in model_response.lower():
        return 'Network'
    elif 'security' in model_response.lower():
        return 'Security'
    else:
        return 'General'

## **Task 1: Ticket Categorization **

In [261]:
# creating a copy of the data
data_1 = data.copy()

In [262]:
#Defining the response funciton for Task 1.
def response_1(prompt,review):
    model_output = llm(
      f"""
      Q: {prompt}
      Review: {review}
      A:
      """,
      max_tokens=32,
      stop=["Q:", "\n"],
      temperature=0.01,
      echo=False,
    )

    temp_output = model_output["choices"][0]["text"]

    return temp_output

In [263]:
prompt_1 = """
    You are an AI analyzing IT support tickets. Classify the category of the provided tickets into the following categories:
    - Hardware Support
    - Software Support
    - Network Support
    - Security Support
"""


In [264]:
start = time.time()
data_1['model_response'] = data_1['support_ticket_text'].apply(lambda x: response_1(prompt_1, x))
end = time.time()

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [265]:
print("Time taken ",(end-start))

Time taken  142.32990908622742


In [266]:
data_1['model_response'].head()


,model_response
0,"Based on the provided ticket, the category wo..."
1,This ticket falls under the Hardware Support ...
2,"Based on the provided ticket, the category wo..."
3,"Based on the provided information, this ticke..."
4,"Based on the provided ticket, it falls under ..."


In [267]:
i = 2
print(data_1.loc[i, 'support_ticket_text'])

I've accidentally deleted essential work documents, causing substantial data loss. I understand the need to avoid further actions on my device. Can you please prioritize the data recovery process and guide me through it?


In [268]:
print(data_1.loc[i, 'model_response'])

 Based on the provided ticket, the category would be:


In [269]:
# applying the function to the model response
data_1['category'] = data_1['model_response'].apply(extract_category)
data_1['category'].head()

,category
0,Network
1,Hardware
2,General
3,Network
4,Hardware


In [270]:
data_1['category'].value_counts()

,category
Hardware,10
Network,5
Software,4
General,2


In [271]:
# Normalizing the model_response_parsed column
final_data_1 = data_1.drop('model_response', axis=1)
final_data_1.head()

,support_tick_id,support_ticket_text,category
0,ST2023-006,My internet connection has significantly slowe...,Network
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware
2,ST2023-008,I've accidentally deleted essential work docum...,General
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Network
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware


In [272]:
# Concatinating two dataframes
data_with_parsed_model_output_1 = pd.concat([data_1, model_response_parsed_df_1], axis=1)
data_with_parsed_model_output_1.head()

,support_tick_id,support_ticket_text,model_response,category
0,ST2023-006,My internet connection has significantly slowe...,"Based on the provided ticket, the category wo...",Network
1,ST2023-007,Urgent help required! My laptop refuses to sta...,This ticket falls under the Hardware Support ...,Hardware
2,ST2023-008,I've accidentally deleted essential work docum...,"Based on the provided ticket, the category wo...",General
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,"Based on the provided information, this ticke...",Network
4,ST2023-010,"My smartphone battery is draining rapidly, eve...","Based on the provided ticket, it falls under ...",Hardware


In [273]:
# Dropping model_response and model_response_parsed columns
final_data_1 = data_with_parsed_model_output_1.drop(['model_response'], axis=1)
final_data_1.head()

,support_tick_id,support_ticket_text,category
0,ST2023-006,My internet connection has significantly slowe...,Network
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware
2,ST2023-008,I've accidentally deleted essential work docum...,General
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Network
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware


# Ticket Categorization and Returning Structured Output (Mistral)

In [274]:
# creating a copy of the data
data_2 = data.copy()

In [275]:
# defining the instructions for the model
prompt_2 = """
    You are an AI analyzing IT support tickets. Classify the category of the provided tickets into the following categories:
    - Hardware
    - Software
    - Network
    - Data Restore
    - Security

    Format the output as a JSON object with a single key-value pair as shown below:
    {"Category": "your_category_prediction"}
"""

In [276]:
start = time.time()
data_2['model_response'] = data_2['support_ticket_text'].apply(lambda x: response_1(prompt_2, x))
end = time.time()

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [277]:
data_2['model_response'].head()

,model_response
0,"{""Category"": ""Network""}"
1,"{""Category"": ""Hardware""}"
2,"{""Category"": ""Data Restore""}"
3,"{""Category"": ""Network""}"
4,"{""Category"": ""Hardware""}"


In [278]:
i = 2
print(data_2.loc[i, 'support_ticket_text'])

I've accidentally deleted essential work documents, causing substantial data loss. I understand the need to avoid further actions on my device. Can you please prioritize the data recovery process and guide me through it?


In [279]:
print(data_2.loc[i, 'model_response'])

 {"Category": "Data Restore"}


In [280]:
# applying the function to the model response
data_2['model_response_parsed'] = data_2['model_response'].apply(extract_json_data)
data_2['model_response_parsed'].head()

Extracted JSON: {"Category": "Network"}
Extracted JSON: {"Category": "Hardware"}
Extracted JSON: {"Category": "Data Restore"}
Extracted JSON: {"Category": "Network"}
Extracted JSON: {"Category": "Hardware"}
Extracted JSON: {"Category": "Security"}
Extracted JSON: {"Category": "Software"}
Extracted JSON: {"Category": "Hardware"}
Extracted JSON: {"Category": "Data Restore"}
Extracted JSON: {"Category": "Hardware"}
Extracted JSON: {"Category": "Data Restore"}
Extracted JSON: {"Category": "Hardware"}
Extracted JSON: {"Category": "Hardware", "SubCategory": "Device Failure"}
Extracted JSON: {"Category": "Data Restore"}
Extracted JSON: {"Category": "Hardware"}
Extracted JSON: {"Category": "Network"}
Extracted JSON: {"Category": "Network"}
Extracted JSON: {"Category": "Data Restore"}
Extracted JSON: {"Category": "Data Restore"}
Extracted JSON: {"Category": "Network"}
Extracted JSON: {"Category": "Software"}


,model_response_parsed
0,{'Category': 'Network'}
1,{'Category': 'Hardware'}
2,{'Category': 'Data Restore'}
3,{'Category': 'Network'}
4,{'Category': 'Hardware'}


In [281]:
model_response_parsed_df_2 = pd.json_normalize(data_2['model_response_parsed'])
model_response_parsed_df_2.head()

,Category,SubCategory
0,Network,NaN
1,Hardware,NaN
2,Data Restore,NaN
3,Network,NaN
4,Hardware,NaN


In [282]:
data_with_parsed_model_output_2 = pd.concat([data_2, model_response_parsed_df_2], axis=1)
data_with_parsed_model_output_2.head()

,support_tick_id,support_ticket_text,model_response,model_response_parsed,Category,SubCategory
0,ST2023-006,My internet connection has significantly slowe...,"{""Category"": ""Network""}",{'Category': 'Network'},Network,NaN
1,ST2023-007,Urgent help required! My laptop refuses to sta...,"{""Category"": ""Hardware""}",{'Category': 'Hardware'},Hardware,NaN
2,ST2023-008,I've accidentally deleted essential work docum...,"{""Category"": ""Data Restore""}",{'Category': 'Data Restore'},Data Restore,NaN
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,"{""Category"": ""Network""}",{'Category': 'Network'},Network,NaN
4,ST2023-010,"My smartphone battery is draining rapidly, eve...","{""Category"": ""Hardware""}",{'Category': 'Hardware'},Hardware,NaN


In [283]:
final_data_2 = data_with_parsed_model_output_2.drop(['model_response','model_response_parsed', 'SubCategory'], axis=1)
final_data_2.head()

,support_tick_id,support_ticket_text,Category
0,ST2023-006,My internet connection has significantly slowe...,Network
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware
2,ST2023-008,I've accidentally deleted essential work docum...,Data Restore
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Network
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware


In [284]:
final_data_2['Category'].value_counts()

,Category
Hardware,7
Data Restore,6
Network,5
Software,2
Security,1


## **Task 2: Creating Tags**

In [285]:
# creating a copy of the data
data_3 = data.copy()

In [286]:
def response_3(prompt,ticket,category):
    model_output = llm(
      f"""
      Q: {prompt}
      Support ticket: {ticket}
      Category: {category}
      A:
      """,
      max_tokens=500, #Complete the code to set the maximum number of tokens the model should generate for this task.
      stop=["Q:", "\n"],
      temperature=0.8, #Complete the code to set the value for temperature.
      echo=False,
    )

    temp_output = model_output["choices"][0]["text"]
    final_output = temp_output[temp_output.index('{'):]

    return final_output

In [287]:
prompt_3 = """
You are an AI specialized in analyzing IT support tickets. Your task is to review each ticket's details and identify key metadata tags that summarize the ticket. For each ticket, extract the relevant information such as issue type, urgency level, department, software or hardware involved, resolution status, and any other critical details.

These tags should be more detailed than "Category" and not replicate Category. Be concise, relevant, and standardized to ensure consistency across all tickets. If unable to tag enter an empty string as the value. Ensure that each ticket has a comprehensive yet succinct set of metadata tags.

    Format the output as a JSON object with a single key-value pair. Here is an example of how this should be formatted
    {"Tags": ["network-issue", "low-urgency"]}
"""

**Note**: The output of the model should be in a structured format (JSON format).

In [288]:
start = time.time()
data_3["model_response"]=final_data_2[['support_ticket_text','Category']].apply(lambda x: response_3(prompt_3, x[0],x[1]),axis =1)
end = time.time()

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [289]:
print("Time taken ",end-start)

Time taken  100.80863976478577


In [290]:
data_3['model_response'].head()


,model_response
0,"{""Tags"": [""network-issue"", ""high-urgency""]}"
1,"{""Tags"": [""hardware-issue"", ""critical-urgency""]}"
2,"{""Tags"": [""data-loss"", ""critical-urgency"", ""IT..."
3,"{""Tags"": [""network-issue"", ""wifi"", ""low-urgenc..."
4,"{""Tags"": [""mobile-device"", ""battery-issue"", ""m..."


In [291]:
i = 2
print(data_3.loc[i, 'support_ticket_text'])

I've accidentally deleted essential work documents, causing substantial data loss. I understand the need to avoid further actions on my device. Can you please prioritize the data recovery process and guide me through it?


In [292]:
print(data_3.loc[i, 'model_response'])

{"Tags": ["data-loss", "critical-urgency", "IT-department"]}


In [293]:
# Applying the function to the model response
data_3['model_response_parsed'] = data_3['model_response'].apply(extract_json_data)

Extracted JSON: {"Tags": ["network-issue", "high-urgency"]}
Extracted JSON: {"Tags": ["hardware-issue", "critical-urgency"]}
Extracted JSON: {"Tags": ["data-loss", "critical-urgency", "IT-department"]}
Extracted JSON: {"Tags": ["network-issue", "wifi", "low-urgency"]}
Extracted JSON: {"Tags": ["mobile-device", "battery-issue", "medium-urgency"]}
Extracted JSON: {"Tags": ["account-access", "security-issue", "high-urgency"]}
Extracted JSON: {"Tags": ["performance-degradation", "software-issue", "high-urgency"]}
Extracted JSON: {"Tags": ["blue_screen_error", "hardware_issue", "high_urgency"]}
Extracted JSON: {"Tags": ["data-recovery", "hardware", "high-urgency"]}
Extracted JSON: {"Tags":["graphics-card","gaming-laptop","hardware-issue","performance"]}
Extracted JSON: {"Tags": ["data-loss", "external-storage", "low-urgency"]}
Extracted JSON: {"Tags": ["monitor-issue", "high-urgency"]}
Extracted JSON: {"Tags": ["hardware-damage", "high-urgency", "data-recovery"]}
Extracted JSON: {"Tags": ["

In [294]:
data_3["model_response_parsed"]

,model_response_parsed
0,"{'Tags': ['network-issue', 'high-urgency']}"
1,"{'Tags': ['hardware-issue', 'critical-urgency']}"
2,"{'Tags': ['data-loss', 'critical-urgency', 'IT..."
3,"{'Tags': ['network-issue', 'wifi', 'low-urgenc..."
4,"{'Tags': ['mobile-device', 'battery-issue', 'm..."
5,"{'Tags': ['account-access', 'security-issue', ..."
6,"{'Tags': ['performance-degradation', 'software..."
7,"{'Tags': ['blue_screen_error', 'hardware_issue..."
8,"{'Tags': ['data-recovery', 'hardware', 'high-u..."
9,"{'Tags': ['graphics-card', 'gaming-laptop', 'h..."


In [295]:
# Normalizing the model_response_parsed column
model_response_parsed_df_3 = pd.json_normalize(data_3['model_response_parsed'])
model_response_parsed_df_3.head()

,Tags
0,"[network-issue, high-urgency]"
1,"[hardware-issue, critical-urgency]"
2,"[data-loss, critical-urgency, IT-department]"
3,"[network-issue, wifi, low-urgency]"
4,"[mobile-device, battery-issue, medium-urgency]"


In [296]:
#  remove the [ and ] from the Tags column data
# Convert the 'Tags' column to string type first
model_response_parsed_df_3['Tags'] = model_response_parsed_df_3['Tags'].astype(str)
model_response_parsed_df_3['Tags'] = model_response_parsed_df_3['Tags'].str.replace('[', '').str.replace(']', '')


<ipython-input-296-20ec9811de53>:4: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  model_response_parsed_df_3['Tags'] = model_response_parsed_df_3['Tags'].str.replace('[', '').str.replace(']', '')


In [297]:
# Concatinating two dataframes
data_with_parsed_model_output_3 = pd.concat([data_3, model_response_parsed_df_3], axis=1)
data_with_parsed_model_output_3.head()

,support_tick_id,support_ticket_text,model_response,model_response_parsed,Tags
0,ST2023-006,My internet connection has significantly slowe...,"{""Tags"": [""network-issue"", ""high-urgency""]}","{'Tags': ['network-issue', 'high-urgency']}","'network-issue', 'high-urgency'"
1,ST2023-007,Urgent help required! My laptop refuses to sta...,"{""Tags"": [""hardware-issue"", ""critical-urgency""]}","{'Tags': ['hardware-issue', 'critical-urgency']}","'hardware-issue', 'critical-urgency'"
2,ST2023-008,I've accidentally deleted essential work docum...,"{""Tags"": [""data-loss"", ""critical-urgency"", ""IT...","{'Tags': ['data-loss', 'critical-urgency', 'IT...","'data-loss', 'critical-urgency', 'IT-department'"
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,"{""Tags"": [""network-issue"", ""wifi"", ""low-urgenc...","{'Tags': ['network-issue', 'wifi', 'low-urgenc...","'network-issue', 'wifi', 'low-urgency'"
4,ST2023-010,"My smartphone battery is draining rapidly, eve...","{""Tags"": [""mobile-device"", ""battery-issue"", ""m...","{'Tags': ['mobile-device', 'battery-issue', 'm...","'mobile-device', 'battery-issue', 'medium-urge..."


In [298]:
# Dropping model_response and model_response_parsed columns
final_data_3 = data_with_parsed_model_output_3.drop(['model_response','model_response_parsed'], axis=1)
final_data_3.head()

,support_tick_id,support_ticket_text,Tags
0,ST2023-006,My internet connection has significantly slowe...,"'network-issue', 'high-urgency'"
1,ST2023-007,Urgent help required! My laptop refuses to sta...,"'hardware-issue', 'critical-urgency'"
2,ST2023-008,I've accidentally deleted essential work docum...,"'data-loss', 'critical-urgency', 'IT-department'"
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,"'network-issue', 'wifi', 'low-urgency'"
4,ST2023-010,"My smartphone battery is draining rapidly, eve...","'mobile-device', 'battery-issue', 'medium-urge..."


In [299]:
# Checking the value counts of Category column
final_data_3['Tags'].value_counts()

,Tags
"'network-issue', 'high-urgency'",2
nan,2
"'hardware-issue', 'critical-urgency'",1
"'data-loss', 'critical-urgency', 'IT-department'",1
"'network-issue', 'wifi', 'low-urgency'",1
"'account-access', 'security-issue', 'high-urgency'",1
"'mobile-device', 'battery-issue', 'medium-urgency'",1
"'blue_screen_error', 'hardware_issue', 'high_urgency'",1
"'data-recovery', 'hardware', 'high-urgency'",1
"'graphics-card', 'gaming-laptop', 'hardware-issue', 'performance'",1


In [300]:
final_data_3 = pd.concat([final_data_3,final_data_2["Category"]],axis=1)

In [301]:
final_data_3 = final_data_3[["support_tick_id","support_ticket_text","Category","Tags"]]
final_data_3

,support_tick_id,support_ticket_text,Category,Tags
0,ST2023-006,My internet connection has significantly slowe...,Network,"'network-issue', 'high-urgency'"
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware,"'hardware-issue', 'critical-urgency'"
2,ST2023-008,I've accidentally deleted essential work docum...,Data Restore,"'data-loss', 'critical-urgency', 'IT-department'"
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Network,"'network-issue', 'wifi', 'low-urgency'"
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware,"'mobile-device', 'battery-issue', 'medium-urge..."
5,ST2023-011,I'm locked out of my online banking account an...,Security,"'account-access', 'security-issue', 'high-urge..."
6,ST2023-012,"My computer's performance is sluggish, severel...",Software,"'performance-degradation', 'software-issue', '..."
7,ST2023-013,I'm experiencing a recurring blue screen error...,Hardware,"'blue_screen_error', 'hardware_issue', 'high_u..."
8,ST2023-014,My external hard drive isn't being recognized ...,Data Restore,"'data-recovery', 'hardware', 'high-urgency'"
9,ST2023-015,The graphics card in my gaming laptop seems to...,Hardware,"'graphics-card', 'gaming-laptop', 'hardware-is..."


## **Task 3: Assigning Priority and ETA**

In [302]:
# creating a copy of the data
data_4 = data.copy()

In [303]:
def response_4(prompt,ticket,category,tags):
    model_output = llm(
      f"""
      Q: {prompt}
      Support ticket: {ticket}
      Category: {category}
      Tags: {tags}
      A:
      """,
      max_tokens=400,  #Complete the code to set the maximum number of tokens the model should generate for this task.
      stop=["Q:", "\n"],
      temperature=0.7, #Complete the code to set the value for temperature.
      echo=False,
    )

    temp_output = model_output["choices"][0]["text"]
    final_output = temp_output[temp_output.index('{'):]

    return final_output

In [304]:
prompt_4 = """
You are an AI specialized in analyzing IT support tickets. Your task is to review each ticket's details and identify both the ticket priority and the estimated time to address the issue (ETA). For each ticket, extract the relevant information such as issue type, urgency level, department, software or hardware involved, resolution status, and any other critical details.

Your primary focus is to determine the ticket priority (e.g., "high", "medium", "low") and estimate the time it will take to address the issue (e.g., "2 hours", "1 day"). If unable to determine the priority or ETA, enter an empty string as the value. Ensure that the extracted priority and ETA are accurate and consistent across all tickets.

Format the output as a JSON object with two key-value pairs. Here is an example of how this should be formatted:
{"Priority": "high", "ETA": "2 hours"}
"""



In [305]:
# Applying generate_llama_response function on support_ticket_text column
start = time.time()
data_4['model_response'] = final_data_3[['support_ticket_text','Category','Tags']].apply(lambda x: response_4(prompt_4, x[0],x[1],x[2]),axis=1)
end = time.time()

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [306]:
print("Time taken ",(end-start))

Time taken  86.35537004470825


In [307]:
data_4['model_response'].head()


,model_response
0,"{""Priority"": ""high"", ""ETA"": ""4 hours""}"
1,"{""Priority"": ""high"", ""ETA"": ""4 hours""}"
2,"{""Priority"": ""high"", ""ETA"": ""4 hours""}"
3,"{""Priority"": ""medium"", ""ETA"": ""4 hours""}"
4,"{""Priority"": ""medium"", ""ETA"": ""4 hours""}"


In [308]:
i = 2
print(data_4.loc[i, 'support_ticket_text'])

I've accidentally deleted essential work documents, causing substantial data loss. I understand the need to avoid further actions on my device. Can you please prioritize the data recovery process and guide me through it?


In [309]:
print(data_4.loc[i, 'model_response'])

{"Priority": "high", "ETA": "4 hours"}


In [310]:
# Applying the function to the model response
data_4['model_response_parsed'] = data_4['model_response'].apply(extract_json_data)
data_4['model_response_parsed'].head()

Extracted JSON: {"Priority": "high", "ETA": "4 hours"}
Extracted JSON: {"Priority": "high", "ETA": "4 hours"}
Extracted JSON: {"Priority": "high", "ETA": "4 hours"}
Extracted JSON: {"Priority": "medium", "ETA": "4 hours"}
Extracted JSON: {"Priority": "medium", "ETA": "4 hours"}
Extracted JSON: {"Priority": "high", "ETA": "1 hour"}
Extracted JSON: {"Priority": "high", "ETA": "4 hours"}
Extracted JSON: {"Priority": "high", "ETA": "4 hours"}
Extracted JSON: {"Priority": "high", "ETA": "4 hours"}
Extracted JSON: {"Priority": "High", "ETA": "4 hours"}
Extracted JSON: {"Priority": "medium", "ETA": "4 hours"}
Extracted JSON: {"Priority": "high", "ETA": "2 hours"}
Extracted JSON: {"Priority": "high", "ETA": "4 hours to a day, depending on the extent of water damage and availability of replacement parts"}
Extracted JSON: {"Priority": "high", "ETA": "4 hours"}
Extracted JSON: {"Priority": "medium", "ETA": "4 hours"}
Extracted JSON: {"Priority": "high", "ETA": "4 hours"}
Extracted JSON: {"Priorit

,model_response_parsed
0,"{'Priority': 'high', 'ETA': '4 hours'}"
1,"{'Priority': 'high', 'ETA': '4 hours'}"
2,"{'Priority': 'high', 'ETA': '4 hours'}"
3,"{'Priority': 'medium', 'ETA': '4 hours'}"
4,"{'Priority': 'medium', 'ETA': '4 hours'}"


In [311]:
# Normalizing the model_response_parsed column
model_response_parsed_df_4 = pd.json_normalize(data_4['model_response_parsed'])
model_response_parsed_df_4.head(21)

,Priority,ETA
0,high,4 hours
1,high,4 hours
2,high,4 hours
3,medium,4 hours
4,medium,4 hours
5,high,1 hour
6,high,4 hours
7,high,4 hours
8,high,4 hours
9,High,4 hours


In [312]:
# Concatinating two dataframes
data_with_parsed_model_output_4 = pd.concat([data_4, model_response_parsed_df_4], axis=1)
data_with_parsed_model_output_4.head()

,support_tick_id,support_ticket_text,model_response,model_response_parsed,Priority,ETA
0,ST2023-006,My internet connection has significantly slowe...,"{""Priority"": ""high"", ""ETA"": ""4 hours""}","{'Priority': 'high', 'ETA': '4 hours'}",high,4 hours
1,ST2023-007,Urgent help required! My laptop refuses to sta...,"{""Priority"": ""high"", ""ETA"": ""4 hours""}","{'Priority': 'high', 'ETA': '4 hours'}",high,4 hours
2,ST2023-008,I've accidentally deleted essential work docum...,"{""Priority"": ""high"", ""ETA"": ""4 hours""}","{'Priority': 'high', 'ETA': '4 hours'}",high,4 hours
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,"{""Priority"": ""medium"", ""ETA"": ""4 hours""}","{'Priority': 'medium', 'ETA': '4 hours'}",medium,4 hours
4,ST2023-010,"My smartphone battery is draining rapidly, eve...","{""Priority"": ""medium"", ""ETA"": ""4 hours""}","{'Priority': 'medium', 'ETA': '4 hours'}",medium,4 hours


In [313]:
# Dropping model_response and model_response_parsed columns
final_data_4 = data_with_parsed_model_output_4.drop(['model_response','model_response_parsed'], axis=1)
final_data_4.head()

,support_tick_id,support_ticket_text,Priority,ETA
0,ST2023-006,My internet connection has significantly slowe...,high,4 hours
1,ST2023-007,Urgent help required! My laptop refuses to sta...,high,4 hours
2,ST2023-008,I've accidentally deleted essential work docum...,high,4 hours
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,medium,4 hours
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",medium,4 hours


In [314]:
final_data_4 = pd.concat([final_data_4,final_data_3[["Category","Tags"]]],axis=1)

In [315]:
final_data_4 = final_data_4[["support_tick_id","support_ticket_text","Category","Tags","Priority","ETA"]]

In [316]:
final_data_4

,support_tick_id,support_ticket_text,Category,Tags,Priority,ETA
0,ST2023-006,My internet connection has significantly slowe...,Network,"'network-issue', 'high-urgency'",high,4 hours
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware,"'hardware-issue', 'critical-urgency'",high,4 hours
2,ST2023-008,I've accidentally deleted essential work docum...,Data Restore,"'data-loss', 'critical-urgency', 'IT-department'",high,4 hours
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Network,"'network-issue', 'wifi', 'low-urgency'",medium,4 hours
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware,"'mobile-device', 'battery-issue', 'medium-urge...",medium,4 hours
5,ST2023-011,I'm locked out of my online banking account an...,Security,"'account-access', 'security-issue', 'high-urge...",high,1 hour
6,ST2023-012,"My computer's performance is sluggish, severel...",Software,"'performance-degradation', 'software-issue', '...",high,4 hours
7,ST2023-013,I'm experiencing a recurring blue screen error...,Hardware,"'blue_screen_error', 'hardware_issue', 'high_u...",high,4 hours
8,ST2023-014,My external hard drive isn't being recognized ...,Data Restore,"'data-recovery', 'hardware', 'high-urgency'",high,4 hours
9,ST2023-015,The graphics card in my gaming laptop seems to...,Hardware,"'graphics-card', 'gaming-laptop', 'hardware-is...",High,4 hours


## **Task 4 - Creating a Draft Response**

In [384]:
# creating a copy of the data
data_5 = data.copy()

In [385]:
def response_5(prompt,ticket,category,tags,priority,eta):
    model_output = llm(
      f"""
      Q: {prompt}
      Support ticket: {ticket}
      Category : {category}
      Tags : {tags}
      Priority: {priority}
      ETA: {eta}
      A:
      """,
      max_tokens=400,
      stop=["Q:", "\n"],
      temperature=0.05, #this is set low as we want the messages to be relatively consistent/deterministic for the end users
      echo=False,
    )

    temp_output = model_output["choices"][0]["text"]


    return temp_output

In [386]:
prompt_5 = """
You are an AI specialized in analyzing IT support tickets. Your task is to review the details of a specific IT support ticket from the provided data and create a professional response to the user who submitted the ticket.

For each ticket, generate a concise, professional, and ethically sound response that acknowledges the issue, provides any relevant details about the next steps, and reassures the user that their request is being handled appropriately. Ensure that the response is accurate, consistent, and respectful.

Here is an example of the data fields to include and how the response message should be formatted:
"Thank you for submitting your IT support request. The IT Department has done a preliminary review of the information you provided and made the following assignments to your issue:
Description: {ticket}, Category: {category}, Priority: {priority}, Estimated time to complete: {eta}. We will be in touch with you shortly to discuss the next steps."


IMPORTANT: there always MUST be a response in the above format if at all possible. If unable to create a response, enter the following text: "Thank you for submitting your IT support request. We have received your issue and are currently reviewing it. We will provide an update as soon as possible. Please feel free to reach out if you need further assistance."
You always need to create a response, blanks or empty strings are not acceptable.

Add a signature as follows:
"Sincerely,
Your IT Support Team"
"""


In [387]:
#Applying generate_llama_response function on support_ticket_text column
start = time.time()
data_5['model_response'] = final_data_4[['support_ticket_text','Category','Tags','Priority','ETA']].apply(lambda x: response_5(prompt_5, x[0],x[1],x[2],x[3],x[4]),axis=1)
end = time.time()

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [388]:
print("Time taken",(end-start))

Time taken 128.58546018600464


In [389]:
data_5.head()


,support_tick_id,support_ticket_text,model_response
0,ST2023-006,My internet connection has significantly slowe...,
1,ST2023-007,Urgent help required! My laptop refuses to sta...,
2,ST2023-008,I've accidentally deleted essential work docum...,Thank you for submitting your IT support requ...
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Thank you for submitting your IT support requ...
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Thank you for submitting your IT support requ...


In [390]:
i = 2
print(data_5.loc[i, 'support_ticket_text'])

I've accidentally deleted essential work documents, causing substantial data loss. I understand the need to avoid further actions on my device. Can you please prioritize the data recovery process and guide me through it?


In [391]:
print(data_5.loc[i, 'model_response'])

 Thank you for submitting your IT support request. The IT Department has done a preliminary review of the information you provided and made the following assignments to your issue:


In [392]:
final_data_5 = pd.concat([final_data_4,data_5["model_response"]],axis=1)

In [393]:
final_data_5.rename(columns={"model_response":"Response"},inplace=True)

In [394]:
final_data_5

,support_tick_id,support_ticket_text,Category,Tags,Priority,ETA,Response
0,ST2023-006,My internet connection has significantly slowe...,Network,"'network-issue', 'high-urgency'",high,4 hours,
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware,"'hardware-issue', 'critical-urgency'",high,4 hours,
2,ST2023-008,I've accidentally deleted essential work docum...,Data Restore,"'data-loss', 'critical-urgency', 'IT-department'",high,4 hours,Thank you for submitting your IT support requ...
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Network,"'network-issue', 'wifi', 'low-urgency'",medium,4 hours,Thank you for submitting your IT support requ...
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware,"'mobile-device', 'battery-issue', 'medium-urge...",medium,4 hours,Thank you for submitting your IT support requ...
5,ST2023-011,I'm locked out of my online banking account an...,Security,"'account-access', 'security-issue', 'high-urge...",high,1 hour,"""Thank you for submitting your IT support req..."
6,ST2023-012,"My computer's performance is sluggish, severel...",Software,"'performance-degradation', 'software-issue', '...",high,4 hours,
7,ST2023-013,I'm experiencing a recurring blue screen error...,Hardware,"'blue_screen_error', 'hardware_issue', 'high_u...",high,4 hours,Thank you for submitting your IT support requ...
8,ST2023-014,My external hard drive isn't being recognized ...,Data Restore,"'data-recovery', 'hardware', 'high-urgency'",high,4 hours,Thank you for submitting your IT support requ...
9,ST2023-015,The graphics card in my gaming laptop seems to...,Hardware,"'graphics-card', 'gaming-laptop', 'hardware-is...",High,4 hours,


## **Model Output Analysis**

In [395]:

final_data = final_data_5.copy()

In [396]:
final_data['Category'].value_counts()

,Category
Hardware,7
Data Restore,6
Network,5
Software,2
Security,1


In [397]:
final_data["Priority"].value_counts()

,Priority
high,13
medium,5
High,3


In [398]:
final_data["ETA"].value_counts()

,ETA
4 hours,18
1 hour,1
2 hours,1
"4 hours to a day, depending on the extent of water damage and availability of replacement parts",1


Let's dive in a bit deeper here.

In [399]:
final_data.groupby(['Category', 'ETA']).support_tick_id.count()

Category      ETA                                                                                            
Data Restore  4 hours                                                                                            6
Hardware      2 hours                                                                                            1
              4 hours                                                                                            5
              4 hours to a day, depending on the extent of water damage and availability of replacement parts    1
Network       4 hours                                                                                            5
Security      1 hour                                                                                             1
Software      4 hours                                                                                            2
Name: support_tick_id, dtype: int64

## **Actionable Insights and Recommendations**


1. **Prioritize Hardware and Software Issues:** Given the high volume of Hardware and Software tickets, allocate more resources to these areas for faster resolution. Consider specialized teams or training to address these categories efficiently.

2. **Standardize ETA Communication:** The wide range of ETAs suggests a lack of consistent estimation practices. Implement guidelines for assigning ETAs based on issue complexity and urgency, improving communication with users and managing expectations.

3. **Streamline Network Issue Resolution:** A significant number of Network tickets have longer ETAs. Investigate the root causes for delays and explore solutions like network optimization or automation to expedite resolution.

4. **Proactive Security Measures:** While Security tickets are fewer, their impact can be critical. Invest in proactive security measures like vulnerability assessments and employee training to prevent future incidents.

5. **Data-Driven Resource Allocation:** Analyze the distribution of ticket categories and ETAs to identify peak periods and allocate resources accordingly. This ensures optimal staffing and minimizes response times.

6. **Continuous Improvement through Feedback:** Implement a feedback mechanism for users to rate their support experience. This valuable data can highlight areas for improvement and enhance overall customer satisfaction.
